# Weather benchmark: 1. Construct the reference outcome table

This notebook constructs the versioned artificial-gap benchmark for the Weather domain. It implements the frozen reference protocol, writes derived outputs only, and never copies external source data into the repository.

**Purpose.** The resulting table combines observable local context, per-method outcomes, and gap provenance. It is the shared input to the descriptive analysis, confirmatory nested evaluation, and final deployment fit in notebooks 2–4.


## Data scope and eligibility

The frozen DWD reference contains hourly `TT_TU` air-temperature series from 16 stations over 2000-01-01 00:00 through 2025-12-31 23:00. Raw DWD archives remain external; the preparation workflow maps DWD missing-value codes and produces local processed station files.

Each station-year contributes four non-overlapping artificial gaps in each of five strata: 1–6, 7–24, 25–72, 73–168, and 169–440 hours. This requests 8,320 gaps before explicit eligibility exclusions.


## Reproducible inputs and deliberate rebuilds

`WEATHER_DATA_DIR` in `.env` must point to the required external data root. `WEATHER_BENCHMARK_DIR` can optionally redirect the derived benchmark output; the published default is `benchmarks/weather`.

The frozen protocol is defined in `configs/weather_final.toml`. It fixes sampling, context requirements, duration strata, random seed, and the scale-floor quantile. Set `PREPARE_RAW_DATA = True` only to deliberately recreate the extracted and processed inputs from the external raw archives; otherwise the existing processed station files are reused.


In [1]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent

source_root = str(PROJECT_ROOT / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from gap_imputation_benchmark.paths import load_local_environment, local_path_or_default, project_relative_path_or_label
load_local_environment(override=True)

if not os.environ.get('WEATHER_DATA_DIR'):
    raise RuntimeError("Set WEATHER_DATA_DIR in .env or in the current session.")

DATA_DIR = Path(os.environ['WEATHER_DATA_DIR'])
BENCHMARK_DIR = local_path_or_default(
    'WEATHER_BENCHMARK_DIR', PROJECT_ROOT / 'benchmarks' / 'weather',
)
CONFIG = PROJECT_ROOT / 'configs' / 'weather_final.toml'
SCRIPT = PROJECT_ROOT / 'scripts' / 'build_weather_benchmark.py'
PREPARE_RAW_DATA = False

BENCHMARK_LOCATION = project_relative_path_or_label(
    BENCHMARK_DIR, fallback_label='configured benchmark directory'
)
print('External data: configured external data directory')
print(f'Benchmark output: {BENCHMARK_LOCATION}')


External data: configured external data directory
Benchmark output: benchmarks/weather


## Generate the benchmark

The canonical script evaluates the same artificial-gap geometry for every registered candidate method: forward fill, nearest boundary, linear interpolation, PCHIP, local natural cubic spline, BIC-selected polynomial, template reconstruction, and a calendar-matched seasonal reference. The seasonal reference uses valid prior and future calendar-year segments and aggregates them pointwise by the mean. Each gap retains portable source references and diagnostics needed for review.


In [2]:
import subprocess

execution_env = os.environ.copy()
execution_env['PYTHONPATH'] = source_root + os.pathsep + execution_env.get('PYTHONPATH', '')
command = [
    sys.executable, str(SCRIPT), '--config', str(CONFIG),
    '--data-dir', str(DATA_DIR), '--output-dir', str(BENCHMARK_DIR),
]
if PREPARE_RAW_DATA:
    command.append('--prepare-data')
subprocess.run(command, cwd=PROJECT_ROOT, env=execution_env, check=True)
print(f'Benchmark outputs written to: {BENCHMARK_LOCATION}')


Weather benchmark written: weather (8,283/8,320 gaps)
Benchmark outputs written to: benchmarks/weather


## Review the generated reference tables

`coverage_table.csv` verifies requested, learnable, and excluded gaps at the relevant domain level. `selected_recordings.csv` records the deterministically selected source units; `input_manifest.csv` records input discovery and portable references; `metadata.json` records the data scope and frozen protocol.

Any sampling shortfall or post-sampling exclusion remains explicit in the output files. Review these records before treating a rerun as equivalent to the published reference.


In [3]:
import json
import pandas as pd

coverage = pd.read_csv(BENCHMARK_DIR / 'coverage_table.csv')
selected_recordings = pd.read_csv(BENCHMARK_DIR / 'selected_recordings.csv')
metadata = json.loads((BENCHMARK_DIR / 'metadata.json').read_text(encoding='utf-8'))
display(coverage)
display(selected_recordings.head())
metadata


,station_id,year,successful_gaps,requested_gaps,shortfall
0,232,2000,20,20,0
1,232,2001,20,20,0
2,232,2002,20,20,0
3,232,2003,20,20,0
4,232,2004,20,20,0
...,...,...,...,...,...
411,5792,2021,20,20,0
412,5792,2022,20,20,0
413,5792,2023,20,20,0
414,5792,2024,19,20,1


,station_id,source_file,first_timestamp,last_timestamp,n_hourly_rows,natural_missing_hours
0,232,data/weather/processed/temperature_2000_2025/p...,2000-01-01,2025-12-31 23:00:00,227928,503
1,427,data/weather/processed/temperature_2000_2025/p...,2000-01-01,2025-12-31 23:00:00,227928,55
2,1048,data/weather/processed/temperature_2000_2025/p...,2000-01-01,2025-12-31 23:00:00,227928,3
3,1346,data/weather/processed/temperature_2000_2025/p...,2000-01-01,2025-12-31 23:00:00,227928,145
4,1420,data/weather/processed/temperature_2000_2025/p...,2000-01-01,2025-12-31 23:00:00,227928,81


{'artifact_type': 'benchmark',
 'domain': 'weather',
 'workflow': 'weather_benchmark',
 'dataset_id': 'DWD_hourly_temperature',
 'variable': 'TT_TU',
 'period': {'start': '2000-01-01 00:00', 'end': '2025-12-31 23:00'},
 'sampling_unit': 'station_year',
 'stations': 16,
 'years_per_station': 26,
 'requested_gaps': 8320,
 'generated_gaps': 8283,
 'excluded_gaps': 37,
 'gap_strata_hours': [[1, 6], [7, 24], [25, 72], [73, 168], [169, 440]],
 'gaps_per_stratum': 4,
 'method_names': ['forward_fill',
  'nearest_boundary',
  'linear',
  'pchip',
  'local_natural_cubic_spline',
  'polyfit_bic',
  'template',
  'seasonal_periodic'],
 'feature_columns': ['realized_gap_duration_hours',
  'left_context_valid_fraction',
  'right_context_valid_fraction',
  'normalized_boundary_jump',
  'normalized_mean_difference_right_minus_left',
  'local_std_over_scale',
  'local_range_over_scale',
  'normalized_trend_before',
  'normalized_trend_after',
  'normalized_trend_difference',
  'trend_before_r2',
  'tre